# 03 - Silver
## 06 - SCD Reference Tables

### Goal
- implement SCD Type 1 for lookup reference tables
- implement SCD Type 2 for asset_reference to keep history
- use MERGE to apply changes from re-ingested source files

In [0]:
%run ../01_setup/00_config

In [0]:
from pyspark.sql import functions as F

## SCD for reference data

**SCD Type 1** - overwrite on change, keep only the current version. Retired records get `is_active = False`.
Used for: `region_reference`, `market_reference`, `event_type_reference`, `weather_alert_reference`

**SCD Type 2** - keep full history. When a record changes, close the old version and insert a new one.
Used for: `asset_reference` - so you can track when an asset was reclassified or moved to a new region.

## SCD Type 1 - region reference

Same pattern applies to market, event_type, and weather_alert reference tables.

In [0]:
# ensure _reference has is_active column
# run this once to add the column if it does not exist
scd1_tables = [
    (silver_region_reference_table,        "region"),
    (silver_market_reference_table,        "market_type"),
    (silver_event_type_reference_table,    "event_type"),
    (silver_weather_alert_reference_table, "weather_alert_level"),
]

for table, _ in scd1_tables:
    existing_cols = set(spark.table(table).columns)
    if "is_active" not in existing_cols:
        spark.sql(f"ALTER TABLE {table} ADD COLUMN is_active BOOLEAN")
        spark.sql(f"UPDATE {table} SET is_active = true WHERE is_active IS NULL")
        print(f"Added is_active column: {table}")
    else:
        print(f"is_active already exists: {table}")

## SCD Type 1 - market, event_type, weather_alert reference

Same pattern applied to remaining lookup tables.

In [0]:
# region reference
spark.table(bronze_region_reference_table).createOrReplaceTempView("source_region")

spark.sql(f"""
MERGE INTO {silver_region_reference_table} AS target
USING source_region AS source
ON target.region = source.region

-- update changed records
WHEN MATCHED AND (
    target.country_code   != source.country_code OR
    target.operating_zone != source.operating_zone
) THEN UPDATE SET
    target.country_code   = source.country_code,
    target.operating_zone = source.operating_zone,
    target.is_active      = true

-- insert new records
WHEN NOT MATCHED THEN INSERT (
    region, country_code, operating_zone, is_active
) VALUES (
    source.region, source.country_code, source.operating_zone, true
)

-- mark retired records as inactive
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET
    target.is_active = false
""")

print(f"SCD Type 1 MERGE complete: {silver_region_reference_table}")
#display(spark.table(silver_region_reference_table))

# market reference
spark.table(bronze_market_reference_table).createOrReplaceTempView("source_market")

spark.sql(f"""
MERGE INTO {silver_market_reference_table} AS target
USING source_market AS source
ON target.market_type = source.market_type
WHEN MATCHED AND target.description != source.description THEN UPDATE SET
    target.description = source.description,
    target.is_active   = true
WHEN NOT MATCHED THEN INSERT (market_type, description, is_active)
    VALUES (source.market_type, source.description, true)
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET target.is_active = false
""")
print(f"SCD Type 1 MERGE complete: {silver_market_reference_table}")
#display(spark.table(silver_region_reference_table))

# event type reference
spark.table(bronze_event_type_reference_table).createOrReplaceTempView("source_event_type")

spark.sql(f"""
MERGE INTO {silver_event_type_reference_table} AS target
USING source_event_type AS source
ON target.event_type = source.event_type
WHEN MATCHED AND target.category != source.category THEN UPDATE SET
    target.category  = source.category,
    target.is_active = true
WHEN NOT MATCHED THEN INSERT (event_type, category, is_active)
    VALUES (source.event_type, source.category, true)
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET target.is_active = false
""")
print(f"SCD Type 1 MERGE complete: {silver_event_type_reference_table}")
#display(spark.table(silver_region_reference_table))

# weather alert reference
spark.table(bronze_weather_alert_reference_table).createOrReplaceTempView("source_weather_alert")

spark.sql(f"""
MERGE INTO {silver_weather_alert_reference_table} AS target
USING source_weather_alert AS source
ON target.weather_alert_level = source.weather_alert_level
WHEN MATCHED AND target.priority_rank != source.priority_rank THEN UPDATE SET
    target.priority_rank = source.priority_rank,
    target.is_active     = true
WHEN NOT MATCHED THEN INSERT (weather_alert_level, priority_rank, is_active)
    VALUES (source.weather_alert_level, source.priority_rank, true)
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET target.is_active = false
""")
print(f"SCD Type 1 MERGE complete: {silver_weather_alert_reference_table}")
#display(spark.table(silver_region_reference_table))

## SCD Type 2 - asset reference

Keeps full history. Each change creates a new row with `valid_from` set to today.
The previous version gets `valid_to` set to today and `is_current = false`.

In [0]:
# ensure silver_asset_reference has SCD Type 2 columns
# run this once to add columns if they do not exist
existing_cols = set(spark.table(silver_asset_reference_table).columns)

if "valid_from" not in existing_cols:
    spark.sql(f"ALTER TABLE {silver_asset_reference_table} ADD COLUMN valid_from DATE")
    spark.sql(f"UPDATE {silver_asset_reference_table} SET valid_from = current_date() WHERE valid_from IS NULL")
    print("Added valid_from column.")

if "valid_to" not in existing_cols:
    spark.sql(f"ALTER TABLE {silver_asset_reference_table} ADD COLUMN valid_to DATE")
    print("Added valid_to column.")

if "is_current" not in existing_cols:
    spark.sql(f"ALTER TABLE {silver_asset_reference_table} ADD COLUMN is_current BOOLEAN")
    spark.sql(f"UPDATE {silver_asset_reference_table} SET is_current = true WHERE is_current IS NULL")
    print("Added is_current column.")

print("SCD Type 2 columns ready.")

In [0]:
source_asset_df = (
    spark.table(bronze_asset_reference_table)
    .withColumn("asset_id",   F.upper(F.trim(F.col("asset_id"))))
    .withColumn("asset_name", F.initcap(F.trim(F.col("asset_name"))))
    .withColumn("region",     F.upper(F.trim(F.col("region"))))
    .withColumn("asset_type", F.upper(F.trim(F.col("asset_type"))))
    .drop("_rescued_data", "ingestion_ts")
    .dropDuplicates(["asset_id"])
)
source_asset_df.createOrReplaceTempView("source_asset")

# Step 1 - close old versions where any column changed
spark.sql(f"""
MERGE INTO {silver_asset_reference_table} AS target
USING source_asset AS source
ON target.asset_id = source.asset_id AND target.is_current = true

WHEN MATCHED AND (
    target.asset_name != source.asset_name OR
    target.region     != source.region     OR
    target.asset_type != source.asset_type
) THEN UPDATE SET
    target.valid_to    = current_date(),
    target.is_current  = false
""")

# Step 2 - insert new versions for changed records and brand new assets
spark.sql(f"""
MERGE INTO {silver_asset_reference_table} AS target
USING (
    SELECT s.*
    FROM source_asset s
    LEFT JOIN {silver_asset_reference_table} t
        ON s.asset_id = t.asset_id AND t.is_current = true
    WHERE t.asset_id IS NULL
       OR t.asset_name != s.asset_name
       OR t.region     != s.region
       OR t.asset_type != s.asset_type
) AS new_records
ON target.asset_id = new_records.asset_id AND target.is_current = true

WHEN NOT MATCHED THEN INSERT (
    asset_id, asset_name, region, asset_type,
    valid_from, valid_to, is_current
) VALUES (
    new_records.asset_id, new_records.asset_name,
    new_records.region, new_records.asset_type,
    current_date(), NULL, true
)
""")

print(f"SCD Type 2 MERGE complete: {silver_asset_reference_table}")
display(spark.table(silver_asset_reference_table).orderBy("asset_id", "valid_from"))

## Verify SCD results

In [0]:
# show current active records
print("Current active assets:")
spark.table(silver_asset_reference_table).filter(
    F.col("is_current") == True
).show()

# show historical records if any exist
history_count = spark.table(silver_asset_reference_table).filter(
    F.col("is_current") == False
).count()
print(f"Historical (closed) asset records: {history_count}")

# show inactive reference records
for table, key_col in [
    (silver_region_reference_table,        "region"),
    (silver_market_reference_table,        "market_type"),
    (silver_event_type_reference_table,    "event_type"),
    (silver_weather_alert_reference_table, "weather_alert_level"),
]:
    inactive = spark.table(table).filter(F.col("is_active") == False).count()
    total    = spark.table(table).count()
    print(f"{table}: {total} total, {inactive} inactive")

In [0]:
%sql
select * from vattenfall_dev.refined.silver_market_reference